# This file is for pulling in data from WDF files on the Renishaw and Quickly Observing their data

In [7]:
import os
import glob
import json
import pandas as pd
import numpy as np
import traceback
import h5py
from pathlib import Path
from PIL import Image
import struct
from natsort import natsorted

from wdf import Wdf, WdfBlockId, WdfFlags, WdfDataUnit, WdfType, WdfScanType, WdfDataType, WdfSpectrumFlags
from renishawWiRE.wdfReader import WDFReader
# ==========================================
# MONKEY-PATCH: Bypass strict WMAP assertions (Safe for Jupyter/Interactive)
# ==========================================
if not getattr(WDFReader._parse_wmap, '_is_patched', False):
    original_parse_wmap = WDFReader._parse_wmap

    def safe_parse_wmap(self):
        try:
            original_parse_wmap(self)
        except ValueError as e:
            if "WMAP" in str(e):
                self._wmap_warning = str(e)
            else:
                raise e

    # Tag the wrapper so we know it has already been applied
    safe_parse_wmap._is_patched = True
    WDFReader._parse_wmap = safe_parse_wmap
# ==========================================
import matplotlib.pyplot as plt

## Extraction of Data into HDF5 Files from Renishaw

In [14]:
def explore_wdf_contents(file_path):
    """
    Exhaustively extracts and prints all readable data from a WDF file
    using the renishawWiRE package to inform HDF5 schema design.
    """
    path = Path(file_path)
    if not path.is_file():
        print(f"File not found: {path}")
        return

    print(f"\n{'='*60}")
    print(f" WDF DIAGNOSTIC REPORT: {path.name}")
    print(f"{'='*60}\n")

    try:
        reader = WDFReader(str(path))
    except Exception as e:
        print(f"Failed to read file: {e}")
        return

    # --- 1. CORE METADATA ---
    print("📋 [1] CORE METADATA")
    print("-" * 30)
    print(f"  Title:              {reader.title}")
    print(f"  User:               {reader.username}")
    print(f"  Software:           {reader.application_name} v{'.'.join(map(str, reader.application_version))}")
    print(f"  Measurement Type:   {reader.measurement_type.name}")
    print(f"  Scan Type:          {reader.scan_type.name}")
    print(f"  Laser Wavelength:   {reader.laser_length:.2f} nm" if reader.laser_length else "  Laser Wavelength:   Unknown")
    print(f"  Spectra Collected:  {reader.count} / {reader.capacity} (Completed: {reader.is_completed})")
    print(f"  Accumulations:      {reader.accumulation_count}")
    print()

    # --- 2. SPECTRAL DATA ---
    print("📈 [2] SPECTRAL DATA & AXES")
    print("-" * 30)
    print(f"  Points per spec:    {reader.point_per_spectrum}")
    
    if hasattr(reader, 'xdata') and reader.xdata is not None:
        unit = reader.xlist_unit.name if hasattr(reader, 'xlist_unit') else "Unknown"
        print(f"  X-Axis (Shift):     {len(reader.xdata)} points | Range: {reader.xdata.min():.2f} to {reader.xdata.max():.2f} [{unit}]")
    
    if hasattr(reader, 'ydata') and reader.ydata is not None and len(reader.ydata) > 0:
        unit = reader.ylist_unit.name if hasattr(reader, 'ylist_unit') else "Unknown"
        print(f"  Y-Axis (Secondary): {len(reader.ydata)} points [{unit}]")

    if hasattr(reader, 'spectra'):
        print(f"  Data Matrix Shape:  {reader.spectra.shape} " + 
              ("(Y, X, Shift)" if len(reader.spectra.shape) == 3 else "(Sample, Shift)"))
        print(f"  Data Min/Max:       {reader.spectra.min():.2f} / {reader.spectra.max():.2f}")
    print()

    # --- 3. HARDWARE TRACKING & ORIGINS (The "Hidden" Data) ---
    print("📍 [3] HARDWARE TRACKING & ORIGINS")
    print("-" * 30)
    
    # Standard spatial arrays
    for axis, attr in [('X', 'xpos'), ('Y', 'ypos'), ('Z', 'zpos')]:
        if hasattr(reader, attr):
            arr = getattr(reader, attr)
            if np.any(arr != 0): # Only print if there's actual movement recorded
                unit_attr = getattr(reader, f"{attr}_unit", None)
                unit_name = unit_attr.name if unit_attr else "Unknown"
                print(f"  Stage {axis} Array:      {len(arr)} points | Range: {arr.min():.2f} to {arr.max():.2f} [{unit_name}]")

    # Deep dive into the Origin List (Time, Temp, Flags, etc.)
    if hasattr(reader, 'origin_list_header') and reader.origin_list_header:
        print("\n  Advanced Tracked Variables (from ORGN block):")
        for i, origin in enumerate(reader.origin_list_header):
            is_xy, data_type, unit_type, annotation, array = origin
            if array is not None:
                # Time arrays are converted to seconds from start by the WDFReader
                print(f"    -> [{data_type.name}] Unit: {unit_type.name} | Note: '{annotation}' | Shape: {array.shape}")
                if data_type.name == "Time" and len(array) > 1:
                    print(f"       Duration: {array[-1]:.2f} seconds ({array[-1]/60:.2f} mins)")
    print()

    # --- 4. MAPPING SPATIAL DATA ---
    if hasattr(reader, 'map_shape'):
        print("🗺️  [4] MAPPING / GRID INFORMATION")
        print("-" * 30)
        w, h = reader.map_shape
        print(f"  Grid Shape:         {w} x {h} (Width x Height)")
        if hasattr(reader, 'map_info'):
            mi = reader.map_info
            x_u = mi['x_unit'].name if hasattr(mi['x_unit'], 'name') else str(mi['x_unit'])
            y_u = mi['y_unit'].name if hasattr(mi['y_unit'], 'name') else str(mi['y_unit'])
            print(f"  Start Coords:       X: {mi['x_start']:.2f} | Y: {mi['y_start']:.2f}")
            print(f"  Step Size:          X: {mi['x_pad']:.4f} | Y: {mi['y_pad']:.4f}")
            print(f"  Total Span:         X: {mi['x_span']:.2f} [{x_u}] | Y: {mi['y_span']:.2f} [{y_u}]")
        print()

    # --- 5. OPTICAL IMAGE DATA ---
    if hasattr(reader, 'img') and reader.img is not None:
        print("📷 [5] NATIVE WHITELIGHT IMAGE")
        print("-" * 30)
        print(f"  Image Found:        Yes (JPEG format)")
        if hasattr(reader, 'img_dimensions'):
            w, h = reader.img_dimensions
            unit = reader.img_dimension_unit.name if hasattr(reader, 'img_dimension_unit') else "Unknown"
            print(f"  Dimensions:         {w:.2f} x {h:.2f} [{unit}]")
        if hasattr(reader, 'img_origins'):
            x, y = reader.img_origins
            print(f"  Image Origin (XY):  ({x:.2f}, {y:.2f})")
        if hasattr(reader, 'img_cropbox'):
            print(f"  Map Crop Box:       {reader.img_cropbox} (left, upper, right, lower px)")
    else:
        print("📷 [5] NATIVE WHITELIGHT IMAGE")
        print("-" * 30)
        print("  Image Found:        No")

    print(f"\n{'='*60}")
    
    # Close resources
    reader.close()

In [8]:
def convert_wdf_to_h5(wdf_path, output_dir=None):
    """
    Converts a WDF file into the Unified Raman HDF5 Standard, 
    dynamically handling Maps, Point Series, and Single Shots.
    Calculates and stores true acquisition times.
    """
    input_path = Path(wdf_path)
    
    if output_dir is None:
        output_dir = input_path.parent / "processed_h5"
    else:
        output_dir = Path(output_dir)
        
    output_dir.mkdir(parents=True, exist_ok=True)
    h5_path = output_dir / f"{input_path.stem}_unified.h5"

    try:
        reader = WDFReader(str(input_path))
    except Exception as e:
        print(f"[!] Failed to read {input_path.name}: {e}")
        return None

    with h5py.File(h5_path, 'w') as f:
        # ---------------------------------------------------------
        # 1. GLOBAL ROOT ATTRIBUTES (Metadata & Completion Status)
        # ---------------------------------------------------------
        f.attrs['original_filename'] = input_path.name
        f.attrs['title'] = reader.title
        f.attrs['measurement_type'] = reader.measurement_type.name
        f.attrs['scan_type'] = reader.scan_type.name
        f.attrs['laser_wavelength_nm'] = float(reader.laser_length) if reader.laser_length else 0.0
        f.attrs['accumulations'] = reader.accumulation_count
        f.attrs['is_completed'] = reader.is_completed
        f.attrs['spectra_count'] = reader.count

        # ---------------------------------------------------------
        # 2. SPECTRAL DATA
        # ---------------------------------------------------------
        grp_spectra = f.create_group('spectra')
        grp_spectra.create_dataset('wavenumbers', data=reader.xdata, compression='gzip')
        
        spectra_data = reader.spectra
        
        # Standardize matrix shapes
        if spectra_data.ndim == 1:
            # Single Shot: Force into (1, wavenumbers)
            spectra_data = spectra_data.reshape(1, -1)
            dset = grp_spectra.create_dataset('spectral_matrix', data=spectra_data, compression='gzip')
            dset.attrs['description'] = '(sample, wavenumber)'
        elif spectra_data.ndim == 2:
            # Point Series: Already (samples, wavenumbers)
            dset = grp_spectra.create_dataset('spectral_matrix', data=spectra_data, compression='gzip')
            dset.attrs['description'] = '(sample, wavenumber)'
        elif spectra_data.ndim == 3:
            # Grid Map: (Y, X, wavenumbers)
            dset = grp_spectra.create_dataset('spectral_cube', data=spectra_data, compression='gzip')
            dset.attrs['description'] = '(y_grid, x_grid, wavenumber)'

            # Append Map Dimensions if available
            if hasattr(reader, 'map_info'):
                mi = reader.map_info
                for key in ['x_start', 'y_start', 'x_pad', 'y_pad', 'x_span', 'y_span']:
                    if key in mi:
                        dset.attrs[f'map_{key}'] = mi[key]

        # ---------------------------------------------------------
        # 3. HARDWARE TRACKING & ACQUISITION TIME
        # ---------------------------------------------------------
        grp_coords = f.create_group('coordinates')
        
        # Set defaults in case the Time array is missing or empty
        f.attrs['duration_seconds'] = 0.0
        f.attrs['avg_time_per_spectrum'] = 0.0
        
        if hasattr(reader, 'origin_list_header') and reader.origin_list_header:
            for origin in reader.origin_list_header:
                is_xy, data_type, unit_type, annotation, array = origin
                
                if array is not None:
                    dset_coord = grp_coords.create_dataset(data_type.name, data=array, compression='gzip')
                    dset_coord.attrs['unit'] = unit_type.name
                    dset_coord.attrs['annotation'] = annotation

                    # Parse Time array to calculate kinetics and map speeds
                    if data_type.name == "Time" and len(array) > 0:
                        if len(array) > 1:
                            # Time spans from first to last point
                            total_duration = float(array[-1] - array[0])
                            avg_acq_time = total_duration / len(array)
                        else:
                            # Single point scans won't have a delta, duration defaults to 0
                            total_duration = 0.0
                            avg_acq_time = 0.0
                            
                        f.attrs['duration_seconds'] = total_duration
                        f.attrs['avg_time_per_spectrum'] = avg_acq_time

        # ---------------------------------------------------------
        # 4. OPTICAL IMAGES & PERCEIVED DIMENSIONS
        # ---------------------------------------------------------
        if hasattr(reader, 'img') and reader.img is not None:
            grp_optical = f.create_group('optical')
            
            # Convert internal IO Bytes to NumPy array
            img_array = np.array(Image.open(reader.img))
            dset_img = grp_optical.create_dataset('whitelight', data=img_array, compression='gzip')
            
            # Store dimensions and physical origins directly on the image dataset
            if hasattr(reader, 'img_dimensions'):
                w, h = reader.img_dimensions
                dset_img.attrs['physical_width'] = w
                dset_img.attrs['physical_height'] = h
                dset_img.attrs['unit'] = reader.img_dimension_unit.name if hasattr(reader, 'img_dimension_unit') else "Micron"
                
            if hasattr(reader, 'img_origins'):
                x, y = reader.img_origins
                dset_img.attrs['origin_x'] = x
                dset_img.attrs['origin_y'] = y
                
            if hasattr(reader, 'img_cropbox'):
                # (left, upper, right, lower px)
                dset_img.attrs['map_crop_box_px'] = reader.img_cropbox

    reader.close()
    print(f"✅ Successfully created: {h5_path.name}")
    return h5_path

In [3]:
def batch_process_wdfs(input_folder, output_folder=None, recursive=False):
    """
    Process all .wdf files in a folder and convert them to the unified HDF5 standard.
    
    Parameters:
    -----------
    input_folder : str or Path
        Directory containing the .wdf files.
    output_folder : str or Path, optional
        Destination for the .h5 files. If None, creates a 'processed_h5' 
        folder inside the input_folder.
    recursive : bool
        If True, searches for .wdf files in all subdirectories as well.
    """
    input_path = Path(input_folder)
    
    # Set default output directory if none is provided
    if output_folder is None:
        output_path = input_path / "processed_h5"
    else:
        output_path = Path(output_folder)
        
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Gather WDF files
    search_pattern = "**/*.wdf" if recursive else "*.wdf"
    wdf_files = list(input_path.glob(search_pattern))
    
    if not wdf_files:
        print(f"No WDF files found in {input_path}")
        return []
        
    print(f"Found {len(wdf_files)} WDF files. Starting batch conversion...\n")
    
    successful_files = []
    failed_files = []
    
    for wdf_file in wdf_files:
        try:
            # 2. Call the unified HDF5 export function
            convert_wdf_to_h5(wdf_file, output_path)
            
            successful_files.append(wdf_file.name)
            
        except Exception as e:
            print(f"    [!] Failed to process {wdf_file.name}: {type(e).__name__} - {e}")
            # Uncomment the next line if you need deep debugging on failed files
            # traceback.print_exc() 
            failed_files.append(wdf_file.name)
            
    # Print Summary Report
    print(f"\n{'='*50}")
    print("Batch Processing Complete!")
    print(f"Successfully processed: {len(successful_files)}/{len(wdf_files)}")
    
    if failed_files:
        print(f"Failed files ({len(failed_files)}):")
        for f in failed_files:
            print(f"  - {f}")
            
    return successful_files

In [4]:
def append_image_to_h5(h5_path, image_path, dataset_name):
    """Append a standalone image to an existing unified HDF5 file."""
    
    # Load and convert the new image
    img = Image.open(image_path)
    img_array = np.array(img)
    
    # Open the file in 'a' (read/write/create) mode
    with h5py.File(h5_path, 'a') as f:
        # Ensure the optical group exists
        if 'optical' not in f:
            grp_image = f.create_group('optical')
        else:
            grp_image = f['optical']
            
        # Create the new dataset for the image
        dset = grp_image.create_dataset(
            dataset_name, 
            data=img_array, 
            compression='gzip'
        )
        
        # Add relevant metadata as attributes
        dset.attrs['source'] = 'external_camera'
        dset.attrs['width'] = img.width
        dset.attrs['height'] = img.height
        
    print(f"Successfully appended {dataset_name} to {h5_path}")

In [ ]:
explore_wdf_contents("/plastic_surf_10um_loc2.wdf")


 WDF DIAGNOSTIC REPORT: plastic_surf_10um_loc2.wdf

📋 [1] CORE METADATA
------------------------------
  Title:              Single scan measurement 1
  User:               Raman user
  Software:           WiRE v5.7.0.44973
  Measurement Type:   Single
  Scan Type:          Static
  Laser Wavelength:   784.70 nm
  Spectra Collected:  1 / 1 (Completed: True)
  Accumulations:      1

📈 [2] SPECTRAL DATA & AXES
------------------------------
  Points per spec:    1015
  X-Axis (Shift):     1015 points | Range: 279.60 to 1963.84 [RamanShift]
  Y-Axis (Secondary): 1 points [Pixels]
  Data Matrix Shape:  (1015,) (Sample, Shift)
  Data Min/Max:       1190.02 / 3687.90

📍 [3] HARDWARE TRACKING & ORIGINS
------------------------------

  Advanced Tracked Variables (from ORGN block):
    -> [Time] Unit: FileTime | Note: 'Time' | Shape: (1,)
    -> [Flags] Unit: Arbitrary | Note: 'Flags' | Shape: (1,)
    -> [Checksum] Unit: Arbitrary | Note: 'Checksum' | Shape: (1,)

📷 [5] NATIVE WHITELIGHT IMA

In [ ]:
# Folder to Process:
inputFolder = "Bioreactor/20260611"
outputFolder = inputFolder + "/Extracted"

# Process the folders
batch_process_wdfs(inputFolder, outputFolder)

Found 7 WDF files. Starting batch conversion...

✅ Successfully created: 0_Ethanol_50x_Map_Rd2_unified.h5
✅ Successfully created: 0_Ethanol_50x_Map_unified.h5
✅ Successfully created: 1_Ethanol_50x_Map_unified.h5
✅ Successfully created: 2_Ethanol_50x_Map_unified.h5
✅ Successfully created: 3_Ethanol_50x_Map_unified.h5
✅ Successfully created: 5_Ethanol_50x_Map_unified.h5
✅ Successfully created: 4_Ethanol_50x_Map_unified.h5

Batch Processing Complete!
Successfully processed: 7/7


['0_Ethanol_50x_Map_Rd2.wdf',
 '0_Ethanol_50x_Map.wdf',
 '1_Ethanol_50x_Map.wdf',
 '2_Ethanol_50x_Map.wdf',
 '3_Ethanol_50x_Map.wdf',
 '5_Ethanol_50x_Map.wdf',
 '4_Ethanol_50x_Map.wdf']

## Visualize Renishaw Map Data Raw

### Helper Functions for Getting Data out of h5 files

In [10]:
def get_experiment_metadata(h5_path):
    """Extract global experiment and kinetic metadata from the root of the file."""
    with h5py.File(h5_path, 'r') as f:
        return dict(f.attrs.items())

def get_main_spectra(h5_path):
    """
    Extracts the main spectral data, wavenumber axis, and map attributes.
    Returns: (wavenumbers, raman_data, is_map_boolean, map_metadata_dict)
    """
    map_meta = {}
    with h5py.File(h5_path, 'r') as f:
        grp = f['spectra']
        wavenumbers = grp['wavenumbers'][:]
        
        if 'spectral_cube' in grp:
            dset = grp['spectral_cube']
            raman_data = dset[:]
            is_map = True
            map_meta = dict(dset.attrs.items())
        elif 'spectral_matrix' in grp:
            raman_data = grp['spectral_matrix'][:]
            is_map = False
        else:
            raise KeyError("No standard spectral dataset found.")
            
    return wavenumbers, raman_data, is_map, map_meta

def get_coordinates(h5_path):
    """
    Extracts tracking dimensions into a dictionary of arrays and their units.
    Format: {'Spatial_X': {'data': array, 'unit': 'Micron'}, ...}
    """
    coords = {}
    with h5py.File(h5_path, 'r') as f:
        if 'coordinates' in f:
            for key, dset in f['coordinates'].items():
                coords[key] = {
                    'data': dset[:],
                    'unit': dset.attrs.get('unit', 'Unknown')
                }
    return coords

def get_optical_context(h5_path):
    """
    Extracts images and their crucial spatial alignment metadata.
    """
    images = {}
    with h5py.File(h5_path, 'r') as f:
        if 'optical' in f:
            for img_name, dset in f['optical'].items():
                images[img_name] = {
                    'data': dset[:],
                    'meta': dict(dset.attrs.items())
                }
    return images

### Example of pulling out data from h5 file

In [ ]:
file_path = "0_Ethanol_50x_Map_unified.h5"

print("--- 1. GLOBAL METADATA & KINETICS ---")
meta = get_experiment_metadata(file_path)
print(f"File:              {meta.get('original_filename')}")
print(f"Measurement Type:  {meta.get('measurement_type')}")
print(f"Laser:             {meta.get('laser_wavelength_nm')} nm")
print(f"Total Duration:    {meta.get('duration_seconds'):.2f} s")
print(f"Acquisition Speed: {meta.get('avg_time_per_spectrum'):.3f} s/point")

print("\n--- 2. RAMAN SPECTRA ---")
wavs, data, is_map, map_meta = get_main_spectra(file_path)
if is_map:
    print(f"Loaded a 3D Spectral Map with shape: {data.shape} (Y, X, Shift)")
    print(f"Map Spatial Span: {map_meta.get('map_span_x')} x {map_meta.get('map_span_y')} microns")
    # Generate a dummy heatmap (e.g., peak intensity)
    heatmap = data.max(axis=2) 
else:
    print(f"Loaded Point Spectra with shape: {data.shape} (Samples, Shift)")
    mean_spectrum = data.mean(axis=0)

print("\n--- 3. HARDWARE TRACKING ---")
coords = get_coordinates(file_path)
for axis, c_dict in coords.items():
    arr = c_dict['data']
    unit = c_dict['unit']
    print(f"Found {axis}: {len(arr)} points | Range: {arr.min():.2f} to {arr.max():.2f} [{unit}]")

print("\n--- 4. OPTICAL IMAGES & ALIGNMENT ---")
images = get_optical_context(file_path)
for name, img_dict in images.items():
    img_array = img_dict['data']
    i_meta = img_dict['meta']
    print(f"Loaded '{name}': {img_array.shape}")
    print(f"  -> Physical Size: {i_meta.get('physical_width'):.2f} x {i_meta.get('physical_height'):.2f} {i_meta.get('unit')}")
    print(f"  -> Origin (X,Y):  ({i_meta.get('origin_x'):.2f}, {i_meta.get('origin_y'):.2f})")
    
    # Optional: Preview image
    # Image.fromarray(img_array).show()

--- 1. GLOBAL METADATA & KINETICS ---
File:              0_Ethanol_50x_Map.wdf
Measurement Type:  Mapping
Laser:             784.701448302926 nm
Total Duration:    1525.31 s
Acquisition Speed: 15.253 s/point

--- 2. RAMAN SPECTRA ---
Loaded a 3D Spectral Map with shape: (10, 10, 1015) (Y, X, Shift)
Map Spatial Span: None x None microns

--- 3. HARDWARE TRACKING ---
Found Checksum: 100 points | Range: 0.00 to 0.00 [Arbitrary]
Found Flags: 100 points | Range: 0.00 to 0.00 [Arbitrary]
Found Spatial_X: 100 points | Range: 15443.36 to 15533.36 [Micron]
Found Spatial_Y: 100 points | Range: -7858.22 to -7768.22 [Micron]
Found Time: 100 points | Range: 0.00 to 1525.31 [FileTime]

--- 4. OPTICAL IMAGES & ALIGNMENT ---
Loaded 'whitelight': (1328, 2352, 3)
  -> Physical Size: 170.98 x 93.43 Micron
  -> Origin (X,Y):  (15375.63, -7892.57)
